# 📦 00 — Config commune & variable cible (Y)

Construit le socle partagé par toute l'équipe : config (`DEPTS`, années,
chemins, depuis `src/config.py`), `dim_temps` et `fact_urgences` (nos 3
cibles Y : allergie / asthme / bronchiolite).

À lancer en premier, avant les notebooks `01x_pipeline_*.py`. Chacun
construit sa table `dim_*.parquet` de son côté, mais tout le monde doit
utiliser les mêmes clés de jointure (`dept`, `annee_mois`), d'où ce socle
commun. Une fois les tables `dim_*` prêtes, `02_merge_final.ipynb` les
fusionne automatiquement à `fact_urgences`.

In [6]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


---
## 1. `dim_temps` — Dimension temporelle

Pas de source externe, tout est calculé à partir de `ANNEE_DEBUT`/`ANNEE_FIN`.

Cette base nous permet de créer les indicateurs temporels.

Colonnes produites :

| Colonne | |
|---|---|
| `annee_mois` | Période, format `YYYY-MM`, clé de jointure |
| `annee`, `mois`, `trimestre`, `semestre` | Dérivés du calendrier |
| `sin_mois`, `cos_mois` | Encodage cyclique du mois (évite la rupture de décembre à janvier) |
| `est_hiver`, `est_printemps`, `est_ete`, `est_automne` | Flags saison |
| `saison_pollen` | 1 pendant la période de pollinisation |
| `flag_covid` | 1 pendant la période Covid (~03/2020 à 04/2022) |

In [7]:
def build_dim_temps() -> pd.DataFrame:
    """
    Génère la dimension temporelle : une ligne par mois.
    Clé primaire : annee_mois (format 'YYYY-MM')
    """
    mois_range = pd.period_range(start=f"{ANNEE_DEBUT}-01",
                                  end=f"{ANNEE_FIN}-12", freq="M")
    df = pd.DataFrame({"annee_mois": mois_range.astype(str)})

    df["annee"]      = mois_range.year
    df["mois"]       = mois_range.month
    df["trimestre"]  = ((mois_range.month - 1) // 3) + 1
    df["semestre"]   = ((mois_range.month - 1) // 6) + 1

    # Encodage cyclique (évite la rupture déc → jan dans les modèles linéaires)
    df["sin_mois"]   = np.sin(2 * np.pi * df["mois"] / 12).round(6)
    df["cos_mois"]   = np.cos(2 * np.pi * df["mois"] / 12).round(6)

    # Indicateurs saisonniers (hémisphère nord)
    df["est_hiver"]     = df["mois"].isin([12, 1, 2]).astype(int)
    df["est_printemps"] = df["mois"].isin([3, 4, 5]).astype(int)
    df["est_ete"]       = df["mois"].isin([6, 7, 8]).astype(int)
    df["est_automne"]   = df["mois"].isin([9, 10, 11]).astype(int)

    # Saison pollinique (indicatif France métropolitaine)
    df["saison_pollen"] = df["mois"].isin([3, 4, 5, 6]).astype(int)

    # Flag Covid (peut influer sur les passages aux urgences)
    df["flag_covid"] = (
        ((df["annee"] == 2020) & (df["mois"] >= 3)) |
        (df["annee"] == 2021) |
        ((df["annee"] == 2022) & (df["mois"] <= 3))
    ).astype(int)

    print(f"dim_temps : {df.shape[0]} mois ({df['annee_mois'].iloc[0]} → {df['annee_mois'].iloc[-1]})")
    print(f"Colonnes  : {list(df.columns)}")
    return df


dim_temps = build_dim_temps()
dim_temps.to_parquet(TABLES_DIR / "dim_temps.parquet", index=False)
print(f"\n Sauvegardé → {TABLES_DIR / 'dim_temps.parquet'}")
dim_temps.head(12)

dim_temps : 72 mois (2020-01 → 2025-12)
Colonnes  : ['annee_mois', 'annee', 'mois', 'trimestre', 'semestre', 'sin_mois', 'cos_mois', 'est_hiver', 'est_printemps', 'est_ete', 'est_automne', 'saison_pollen', 'flag_covid']

 Sauvegardé → /Users/siranh/Documents/Data Scientest/projet_liora/data/processed/dim_temps.parquet


,annee_mois,annee,mois,trimestre,semestre,sin_mois,cos_mois,est_hiver,est_printemps,est_ete,est_automne,saison_pollen,flag_covid
0,2020-01,2020,1,1,1,0.500000,0.866025,1,0,0,0,0,0
1,2020-02,2020,2,1,1,0.866025,0.500000,1,0,0,0,0,0
2,2020-03,2020,3,1,1,1.000000,0.000000,0,1,0,0,1,1
3,2020-04,2020,4,2,1,0.866025,-0.500000,0,1,0,0,1,1
4,2020-05,2020,5,2,1,0.500000,-0.866025,0,1,0,0,1,1
5,2020-06,2020,6,2,1,0.000000,-1.000000,0,0,1,0,1,1
6,2020-07,2020,7,3,2,-0.500000,-0.866025,0,0,1,0,0,1
7,2020-08,2020,8,3,2,-0.866025,-0.500000,0,0,1,0,0,1
8,2020-09,2020,9,3,2,-1.000000,-0.000000,0,0,0,1,0,1
9,2020-10,2020,10,4,2,-0.866025,0.500000,0,0,0,1,0,1


---
## 2. 🎯 `fact_urgences` — Variable cible (Y)

Source : Santé Publique France (Odissé).

| Pathologie | URL |
|---|---|
| Allergie | https://odisse.santepubliquefrance.fr/explore/dataset/allergie-passages-aux-urgences-et-actes-sos-medecins-dep/export/ |
| Asthme | https://odisse.santepubliquefrance.fr/explore/dataset/asthme-passages-aux-urgences-et-actes-sos-medecins-dep/export/ |
| Bronchiolite | https://odisse.santepubliquefrance.fr/explore/dataset/bronchiolite-passages-aux-urgences-et-actes-sos-medecins-departement/export/ |

Fichiers attendus dans `data/raw/` : `allergie_urgences.csv`, `asthme_urgences.csv`, `bronchiolite_urgences.csv`.

Colonnes produites (Y = allergie / asthme / bronchiolite) :

| Colonne | |
|---|---|
| `taux_urgences_X` | Part des passages aux urgences pour X, /100k passages urgences toutes causes |
| `taux_hosp_X` | Part des hospitalisations post-urgences pour X, /100k hospitalisations toutes causes |
| `taux_sos_X` | Part des actes SOS Médecins pour X, /100k actes toutes causes |

Ce sont des parts relatives (dénominateur = volume total toutes causes du
même sous-groupe dept/semaine/classe d'âge), pas des taux d'incidence en
population. Ne jamais agréger entre départements ni entre classes d'âge, les
dénominateurs ne sont pas comparables.

In [8]:
for fname in ["asthme_urgences.csv", "bronchiolite_urgences.csv", "allergie_urgences.csv"]:
    df = pd.read_csv(f"data/raw/{fname}", sep=",", nrows=1, dtype=str)
    print(f"\n{fname} :")
    for col in df.columns:
        print(f"  '{col}'")


asthme_urgences.csv :
  '1er jour de la semaine'
  'Semaine'
  'Département Code'
  'Département'
  'Classe d'âge'
  'Taux de passages aux urgences pour asthme'
  'Taux d'hospitalisations après passages aux urgences pour asthme'
  'Taux d'actes médicaux SOS médecins pour asthme'
  'Région Code'
  'Région'

bronchiolite_urgences.csv :
  '1er jour de la semaine'
  'Semaine'
  'Département Code'
  'Département'
  'Classe d'âge'
  'Taux de passages aux urgences pour bronchiolite'
  'Taux d'hospitalisations après passages aux urgences pour bronchiolite'
  'Taux d'actes médicaux SOS médecins pour bronchiolite'
  'Région Code'
  'Région'

allergie_urgences.csv :
  '1er jour de la semaine'
  'Semaine'
  'Département Code'
  'Département'
  'Classe d'âge'
  'Taux de passages aux urgences pour allergie'
  'Taux d'hospitalisations après passages aux urgences pour allergie'
  'Taux d'actes médicaux SOS médecins pour allergie'
  'Région Code'
  'Région'


In [9]:
df = pd.read_csv("data/raw/bronchiolite_urgences.csv", sep=",", dtype=str)
df = df.iloc[:, [0, 2, 4, 5, 6, 7]]
df.columns = ["date", "dept", "classe_age","taux_urgences", "taux_hosp", "taux_sos"]

# Voir toutes les valeurs uniques de classe_age
print(df["classe_age"].unique())

# il n'y a que 0 an dans la colonne classe_age, on peut filtrer dessus pour ne garder que les lignes "Tous âges"

['0 an']


In [10]:
def parse_urgences(pathologie: str, filepath: Path) -> pd.DataFrame:
    """
    Charge et nettoie un fichier CSV de Santé Publique France.

    PARAMÈTRES :
    ─────────────
    pathologie : "allergie", "asthme" ou "bronchiolite"
    filepath   : data/raw/{pathologie}_urgences.csv

    RETOUR :
    ─────────
    DataFrame avec colonnes :
        dept                        → code département (ex: "75")
        annee_mois                  → période (ex: "2021-03")
        taux_urgences_{pathologie}  → nb de passages aux urgences pour la pathologie,
                                      pour 100 000 passages aux urgences avec un diagnostic
                                      codé (toutes causes confondues), même dept/classe d'âge/semaine
        taux_hosp_{pathologie}      → nb d'hospitalisations après passage aux urgences pour la
                                      pathologie, pour 100 000 hospitalisations après passage avec
                                      diagnostic renseigné (toutes causes). Mesure la part de la
                                      pathologie parmi les hospitalisations post-urgences, PAS le
                                      risque d'hospitalisation parmi les patients de la pathologie
        taux_sos_{pathologie}       → nb d'actes SOS Médecins pour la pathologie, pour 100 000
                                      actes SOS Médecins avec diagnostic renseigné (circuit de
                                      soins différent : médecine de ville/domicile, non hospitalier)

    !!! IMPORTANT — Ces 3 taux sont des PARTS RELATIVES (dénominateur = volume total
    toutes causes confondues sur le même sous-groupe), PAS des taux d'incidence en
    population. Santé Publique France précise explicitement qu'ils ne doivent être
    NI agrégés entre départements NI agrégés entre classes d'âge, car le dénominateur
    diffère d'un sous-groupe à l'autre (ex : moyenne de taux_urgences entre deux depts
    n'a pas de sens). Le pipeline respecte déjà cette contrainte : dept reste une clé
    de la table (jamais moyenné) et un seul niveau de classe d'âge est conservé par
    pathologie (cf ÉTAPE 3). Ne pas casser cette invariance dans du code ultérieur.

    Structure commune aux 3 fichiers :
        [0] 1er jour de la semaine  ← date
        [1] Semaine
        [2] Département Code        ← dept
        [3] Département
        [4] Classe d'âge            ← filtre age
        [5] Taux urgences           ← Y principal
        [6] Taux hospitalisations   ← Y secondaire
        [7] Taux SOS médecins       ← Y tertiaire
        [8] Région Code
        [9] Région
    """

    # ── ÉTAPE 1 : Chargement ─────────────────────────────────────────────────
    # dtype=str → tout en texte d'abord pour éviter les erreurs de type mixte
    df = pd.read_csv(filepath, sep=",", encoding="utf-8",
                     dtype=str, low_memory=False)
    print(f"\n[{pathologie.upper()}]")
    print(f"  Lignes brutes       : {len(df):,}")

    # ── ÉTAPE 2 : Sélection et renommage par position ────────────────────────
    # On prend uniquement les 8 premières colonnes utiles
    # et on leur donne des noms clairs tout de suite
    df = df.iloc[:, [0, 2, 4, 5, 6, 7]].copy()
    df.columns = [
        "date",
        "dept",
        "classe_age",
        f"taux_urgences_{pathologie}",
        f"taux_hosp_{pathologie}",
        f"taux_sos_{pathologie}",
    ]

    # ── ÉTAPE 3 : Filtrage sur la classe d'âge globale ───────────────────────
    # Allergie & Asthme  → "Tous âges"  (population complète)
    # Bronchiolite       → "0 an"       (nourrissons uniquement, pas de "Tous âges")
    # On garde les deux valeurs possibles avec isin()
    # !!! Un seul niveau de classe d'âge conservé par pathologie : ne jamais
    # mélanger/agréger "Tous âges" avec "0 an" ou avec les autres tranches
    # (00-14, 15-64, 65+), les dénominateurs ne sont pas comparables entre elles.
    valeurs_age = df["classe_age"].unique().tolist()
    print(f"  Classes d'âge dispo : {valeurs_age}")

    valeurs_valides = ["Tous âges", "0 an"]
    df = df[df["classe_age"].isin(valeurs_valides)].copy()
    print(f"  Après filtre âge : {len(df):,} lignes "
          f"({df['classe_age'].unique().tolist()})")

    # ── ÉTAPE 4 : Convertir la date en mois ──────────────────────────────────
    # La colonne contient le lundi de chaque semaine : "2020-03-30"
    # On la convertit pour extraire l'année et le mois
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    nb_invalides = df["date"].isna().sum() #ok pas de date manquante
    if nb_invalides > 0:
        print(f"  !!!  {nb_invalides} dates manquantes supprimées")
    df = df.dropna(subset=["date"])

    # annee_mois = clé de jointure avec toutes les autres tables
    # Ex : semaines S13, S14, S15 de mars → toutes donnent "2020-03"
    df["annee_mois"] = df["date"].dt.to_period("M").astype(str)
    df["annee"]      = df["date"].dt.year

    # ── ÉTAPE 5 : Filtrage temporel 2020–2025 ────────────────────────────────
    avant = len(df)
    df = df[(df["annee"] >= ANNEE_DEBUT) & (df["annee"] <= ANNEE_FIN)]
    print(f"  Filtrage {ANNEE_DEBUT}–{ANNEE_FIN} : {avant:,} → {len(df):,} lignes")

    # ── ÉTAPE 6 : Nettoyage du code département ───────────────────────────────
    # "9" → "09" | " 68" → "68" | "2a" → "2A"
    df["dept"] = (df["dept"].astype(str).str.strip().str.upper().str.zfill(2))
    avant = len(df)
    df = df[df["dept"].isin(DEPTS)]
    print(f"  Filtrage depts : {avant:,} → {len(df):,} lignes")

    # ── ÉTAPE 7 : Conversion numérique des taux ───────────────────────────────
    # Les valeurs étaient en texte (dtype=str au chargement)
    # errors="coerce" → les "-" ou "n/a" deviennent NaN
    cols_taux = [f"taux_urgences_{pathologie}",
                 f"taux_hosp_{pathologie}",
                 f"taux_sos_{pathologie}"]
    for col in cols_taux:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── ÉTAPE 8 : Agrégation hebdomadaire → mensuelle ─────────────────────────
    # Les données sont hebdomadaires (1 ligne par semaine × département)
    # On prend la MOYENNE des semaines du mois car ce sont des TAUX
    # (pas des comptages → la somme n'aurait pas de sens)
    # !!!  Cette moyenne reste PAR département (dept fait partie du groupby) :
    # on ne moyenne jamais entre départements, seulement entre semaines d'un
    # même mois pour un même département.
    df_agg = (
        df.groupby(["dept", "annee_mois"])[cols_taux]
        .mean() #TODO : vérifier si on veut mean() ou max() pour les taux
        .round(2)
        .reset_index()
    )

    print(f"  Résultat      : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(f"  Période       : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"  Départements  : {df_agg['dept'].nunique()}")
    return df_agg


# ══════════════════════════════════════════════════════════════════════════════
# CHARGEMENT DES 3 PATHOLOGIES
# ══════════════════════════════════════════════════════════════════════════════

FICHIERS_URGENCES = {
    "allergie":     "allergie_urgences.csv",
    "asthme":       "asthme_urgences.csv",
    "bronchiolite": "bronchiolite_urgences.csv",
}

dfs_patho = {}
for patho, fname in FICHIERS_URGENCES.items():
    fpath = RAW_DIR / fname
    if fpath.exists():
        dfs_patho[patho] = parse_urgences(patho, fpath)
    else:
        print(f"!!!  Fichier manquant : {fname}")


# ══════════════════════════════════════════════════════════════════════════════
# FUSION EN UNE SEULE TABLE fact_urgences
# ══════════════════════════════════════════════════════════════════════════════

if dfs_patho:

    # Tableau complet : 96 depts × 72 mois = 6 912 lignes garanties
    dim_t = pd.read_parquet(TABLES_DIR / "dim_temps.parquet")[["annee_mois"]] # On ne prend que la clé de jointure : annee_mois
    skeleton = pd.MultiIndex.from_product(
        [DEPTS, dim_t["annee_mois"]],
        names=["dept", "annee_mois"]
    ).to_frame(index=False) 
    # On obtient pour un département les mois du 2020-01 au 2025-12, même s'il existe des NaNs.

    # Jointure de chaque pathologie sur le squelette
    fact = skeleton.copy()
    for patho, df_tmp in dfs_patho.items():
        fact = fact.merge(df_tmp, on=["dept", "annee_mois"], how="left")

    # Sauvegarde
    fact.to_parquet(TABLES_DIR / "fact_urgences.parquet", index=False)
    print(f"   {fact.shape[0]:,} lignes × {fact.shape[1]} colonnes")
    display(fact.head(10))

    # Rapport de couverture
    print("\n📊 Couverture finale :")
    for col in [c for c in fact.columns if c.startswith("taux_")]:
        pct = fact[col].notna().mean() * 100
        statut = "✅" if pct > 80 else "⚠️"
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"  {statut} {col:<40} {barre} {pct:.1f}%")


[ALLERGIE]
  Lignes brutes       : 136,864
  Classes d'âge dispo : ['65 ans ou plus', 'Tous âges', '00-14 ans', '15-64 ans']
  Après filtre âge : 34,216 lignes (['Tous âges'])
  Filtrage 2020–2025 : 34,216 → 32,552 lignes
  Filtrage depts : 32,552 → 30,048 lignes
  Résultat      : 6,912 lignes × 5 colonnes
  Période       : 2020-01 → 2025-12
  Départements  : 96

[ASTHME]
  Lignes brutes       : 136,864
  Classes d'âge dispo : ['00-14 ans', '15-64 ans', '65 ans ou plus', 'Tous âges']
  Après filtre âge : 34,216 lignes (['Tous âges'])
  Filtrage 2020–2025 : 34,216 → 32,552 lignes
  Filtrage depts : 32,552 → 30,048 lignes
  Résultat      : 6,912 lignes × 5 colonnes
  Période       : 2020-01 → 2025-12
  Départements  : 96

[BRONCHIOLITE]
  Lignes brutes       : 34,216
  Classes d'âge dispo : ['0 an']
  Après filtre âge : 34,216 lignes (['0 an'])
  Filtrage 2020–2025 : 34,216 → 32,552 lignes
  Filtrage depts : 32,552 → 30,048 lignes
  Résultat      : 6,912 lignes × 5 colonnes
  Période   

,dept,annee_mois,taux_urgences_allergie,taux_hosp_allergie,taux_sos_allergie,taux_urgences_asthme,taux_hosp_asthme,taux_sos_asthme,taux_urgences_bronchiolite,taux_hosp_bronchiolite,taux_sos_bronchiolite
0,01,2020-01,815.39,245.12,NaN,402.03,878.26,NaN,23287.66,44861.11,NaN
1,01,2020-02,722.06,228.66,NaN,462.67,437.93,NaN,10169.56,25875.35,NaN
2,01,2020-03,377.48,193.97,NaN,805.16,841.07,NaN,9939.19,14230.77,NaN
3,01,2020-04,500.03,98.04,NaN,574.89,863.81,NaN,0.00,0.00,NaN
4,01,2020-05,568.77,311.94,NaN,522.41,552.69,NaN,0.00,0.00,NaN
5,01,2020-06,796.83,585.61,NaN,272.48,368.71,NaN,2222.22,12500.00,NaN
6,01,2020-07,1211.83,323.65,NaN,345.01,398.95,NaN,2380.95,4166.67,NaN
7,01,2020-08,1335.92,312.04,NaN,343.45,514.11,NaN,833.33,0.00,NaN
8,01,2020-09,842.45,284.92,NaN,381.14,627.62,NaN,3010.88,9166.67,NaN
9,01,2020-10,652.28,300.14,NaN,461.62,594.91,NaN,3229.81,2777.78,NaN



📊 Couverture finale :
  ✅ taux_urgences_allergie                   ██████████ 100.0%
  ✅ taux_hosp_allergie                       █████████░ 99.9%
  ⚠️ taux_sos_allergie                        ████░░░░░░ 46.5%
  ✅ taux_urgences_asthme                     ██████████ 100.0%
  ✅ taux_hosp_asthme                         █████████░ 99.9%
  ⚠️ taux_sos_asthme                          ████░░░░░░ 46.5%
  ✅ taux_urgences_bronchiolite               █████████░ 100.0%
  ✅ taux_hosp_bronchiolite                   █████████░ 96.6%
  ⚠️ taux_sos_bronchiolite                    ████░░░░░░ 46.4%
